# Forecast an opposing NN pilot's future maneuvers using the observer's current skill

Run **12_prepare_nn_duel_skill_forecasting.ipynb** first. This notebook ingests its
verified dataset and trains new neural networks using the same training machinery
as **09b_paired_skill_forecasting.ipynb**, with the expanded input projection.
For A observing B, inputs contain **A's 54 observation/acceleration features plus
A's current catalogue-skill indicators**; all three output heads predict **B's future
skill**, at **+1, +2 and +5 seconds**. Both directions share the same model.

**Run All trains three independent networks (seeds 42, 43 and 44) by default**, each
for up to 100 epochs with the 09b early-stopping policy. Change `TRAINING_SEEDS` to
run fewer or more replicas. Architecture: two Transformer layers, width 192,
six attention heads, dropout 0.1, AdamW, masked equal-horizon cross-entropy and
gradient clipping. A single network has three horizon heads, rather than three
independent horizon models. No results or trained weights are bundled here.

Each run uses a fresh timestamped folder and preserves every strict improvement in
mean **test** NLL. As in 09b, **test is the model-selection partition**; validation
is held out until training and checkpoint selection have finished for every seed.
Validation results are descriptive and must not be used to select seeds or tune
settings. Catalogue selections are maneuver labels, not proof of distinct physical execution.

Install `.[ml]` and a Jupyter/IPython kernel first. CUDA uses BF16 when supported;
CPU uses FP32. The default streaming data path avoids caching the approximately
13.7 GiB full feature array on a 16 GiB GPU. Full training can take substantial
time and disk space. Every code cell is timed; GPU work is synchronized.

For a Windows PyArrow import failure, see the [runtime repair instructions](../docs/nn_duel_skill_forecasting.md#windows-pyarrow-import-failure).


In [ ]:
%%time
from pathlib import Path
import os
import sys

root_override = os.environ.get("BVR_NN_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in candidates
             if (p / "bvr_behavior_prediction/nn_duel_skill_forecasting.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set BVR_NN_ROOT to the full Trajectory_Classification checkout path.")
sys.path.insert(0, str(ROOT))
from bvr_behavior_prediction.notebook_runtime import require_pyarrow
require_pyarrow()  # Report the kernel environment and native-library repair steps.
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from bvr_behavior_prediction import nn_duel_skill_forecasting as duel_data
from bvr_behavior_prediction.paired_skill_forecasting import (
    HORIZONS, HISTORY_STEPS, RAW_HISTORY_STEPS, SPLITS, read_json, write_json, sha256,
)
EXPERIMENT_ROOT = ROOT / "artifacts/nn_duel_skill_forecasting"
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

import torch
from bvr_behavior_prediction.paired_skill_forecasting import (
    DEFAULT_TRAINING, train_model, final_evaluation, metric_tables, resolve_precision,
)
from bvr_behavior_prediction.paired_skill_tracking import log_run_artifacts

DATA_OVERRIDE = os.environ.get("BVR_NN_FORECAST_DATA")
if not DATA_OVERRIDE and not (EXPERIMENT_ROOT / "latest_dataset.json").exists():
    raise FileNotFoundError("Run notebook 12 first, or set BVR_NN_FORECAST_DATA to its dataset directory.")
receipt = None if DATA_OVERRIDE else read_json(EXPERIMENT_ROOT / "latest_dataset.json")
DATA = Path(DATA_OVERRIDE or receipt["directory"]).expanduser().resolve()
if receipt and sha256(DATA / "manifest.json") != receipt["manifest_sha256"]:
    raise ValueError("The prepared dataset changed since notebook 12 completed.")
data = duel_data.NNDuelData(DATA)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TRAINING_SEEDS = [42, 43, 44]  # One independently fitted three-horizon NN per seed.
TRAIN_MODEL = True
RESUME = False
EXISTING_RUNS = {}  # For resume/report-only: {42: r"C:\...\runs\nn-duel-s42_...", ...}.
if not TRAINING_SEEDS or len(set(TRAINING_SEEDS)) != len(TRAINING_SEEDS):
    raise ValueError("Choose at least one distinct training seed.")
if (RESUME or not TRAIN_MODEL) and set(EXISTING_RUNS) != set(TRAINING_SEEDS):
    raise ValueError("Set EXISTING_RUNS for every selected seed when resuming or reporting.")
RUN_STAMP = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ")
RUNS = {seed: Path(EXISTING_RUNS[seed]).expanduser().resolve() if seed in EXISTING_RUNS
        else EXPERIMENT_ROOT / "runs" / f"nn-duel-s{seed}_{RUN_STAMP}"
        for seed in TRAINING_SEEDS}
CONFIG = dict(DEFAULT_TRAINING)
CONFIG.update(epochs=100, early_stopping=True, early_stopping_min_epochs=50,
              early_stopping_patience=10, precision="bf16", batch_size=512,
              evaluation_batch_size=1024, data_backend="prefetch" if DEVICE == "cuda" else "direct",
              num_workers=2, cpu_threads=4, compile_model=False)
MLFLOW_ENABLED = os.environ.get("BVR_NN_FORECAST_MLFLOW", "1") == "1"
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI",
    f"sqlite:///{(EXPERIMENT_ROOT / 'mlflow.db').as_posix()}")
TRACKING = {"enabled": MLFLOW_ENABLED, "tracking_uri": MLFLOW_TRACKING_URI,
            "experiment": "nn-duel-own-skill-forecasting", "log_per_class": True,
            "log_model": True,
            "artifact_root": (EXPERIMENT_ROOT / "mlflow_artifacts").as_uri()
                             if MLFLOW_TRACKING_URI.startswith("sqlite:") else None}
print("Prepared dataset:", DATA)
print("Model run directories:", {s: str(p) for s, p in RUNS.items()})
display(pd.Series({**CONFIG, "device": DEVICE,
    "effective_precision": resolve_precision(DEVICE, CONFIG["precision"]),
    "input_features": len(data.manifest["features"]), "classes": len(data.classes),
    "history_steps": HISTORY_STEPS, "horizons_s": HORIZONS}))
display(pd.DataFrame(data.manifest["counts"]).T)
if MLFLOW_ENABLED and MLFLOW_TRACKING_URI.startswith("sqlite:"):
    print(f'mlflow ui --backend-store-uri "{MLFLOW_TRACKING_URI}" --host 127.0.0.1 --port 5000')
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.cuda.synchronize()


## Fit independently seeded networks

The loss averages the three available horizon cross-entropies with independent
missing-label masks. Scaling and the split are fixed by notebook 12 for every seed.
Training reports train/test loss and per-class F1 each epoch. The first 50 epochs
always run; patience starts at epoch 51 and ten epochs without a new all-time
minimum test NLL stop the run (earliest stop: epoch 60). Disable early stopping
to run all 100 epochs. `RESUME=True` requires the same source, data and configuration.
Checkpoint input dimensions, feature ordering and dataset digest are stored with weights.


In [ ]:
%%time
histories = {}
def epoch_output(row, summary, per_class):
    print(f"Epoch {row['epoch']}: train NLL={row['train_nll']:.5f}, "
          f"test NLL={row['test_nll']:.5f}, {row['seconds']:.1f}s", flush=True)
    display(summary[["split", "horizon_s", "nll", "accuracy", "macro_f1",
                     "supported_macro_f1", "weighted_f1"]].round(4))
    with pd.option_context("display.max_rows", None):
        display(per_class.pivot(index=["horizon_s", "skill"], columns="split",
                                 values=["class_nll", "f1", "support"]).round(4))

for seed, run in RUNS.items():
    print(f"Training seed {seed}: {run}", flush=True)
    config = {**CONFIG, "seed": seed}
    tracking = {**TRACKING, "run_name": run.name}
    if TRAIN_MODEL:
        histories[seed] = train_model(data, run, config, callback=epoch_output,
                                     device=DEVICE, resume=RESUME, tracking=tracking)
    else:
        histories[seed] = pd.read_csv(run / "history.csv")
    status = read_json(run / "training_status.json")
    if status["status"] not in {"completed", "early_stopped"}:
        raise ValueError(f"Training has not concluded for seed {seed}.")
    if read_json(run / "run.json")["training"] != config:
        raise ValueError(f"Requested configuration differs from saved seed {seed}.")
    display(histories[seed].tail())
if DEVICE == "cuda":
    torch.cuda.synchronize()


## Final evaluation and a training-frequency baseline

All seeds have finished selection before this cell accesses validation. The selected
checkpoint of each seed is evaluated on train, test and validation, with per-class
NLL/F1/support, confusion matrices and forecast probabilities saved in its run folder.
A Laplace-smoothed training-frequency baseline uses training labels only.
The default report displays the **first configured seed**, chosen before seeing scores;
no winner is selected using validation.


In [ ]:
%%time
all_metrics = []
for seed, run in RUNS.items():
    best_epoch, summary, scores, results = final_evaluation(
        data, run, DEVICE, tracking={**TRACKING, "run_name": run.name})
    all_metrics.append(summary.assign(seed=seed, selected_epoch=best_epoch))
    display(Markdown(f"**Seed {seed}, test-selected epoch {best_epoch}**"))
    display(summary.round(4))
    del results  # Release full prediction arrays between independently fitted models.
comparison = pd.concat(all_metrics, ignore_index=True)
for run in RUNS.values():
    comparison.to_csv(run / "seed_comparison.csv", index=False)

frequency = np.zeros((3, len(data.classes)), np.float32)
for h in range(len(HORIZONS)):
    labels = data.targets[data.indices["train"], h]
    counts = np.bincount(labels[labels >= 0], minlength=len(data.classes))
    frequency[h] = (counts + 1) / (counts.sum() + len(data.classes))
baseline_rows = []
for split in SPLITS:
    labels = data.targets[data.indices[split]]
    probabilities = np.broadcast_to(frequency, (len(labels), *frequency.shape))
    baseline, _, _ = metric_tables(labels, probabilities, data.classes, split)
    baseline_rows.append(baseline)
baseline_metrics = pd.concat(baseline_rows, ignore_index=True)
for run in RUNS.values():
    baseline_metrics.to_csv(run / "training_frequency_baseline.csv", index=False)
display(baseline_metrics.assign(model="Training frequency").round(4))

REPORT_SEED = TRAINING_SEEDS[0]
RUN = RUNS[REPORT_SEED]
history = histories[REPORT_SEED]
best_epoch = read_json(RUN / "selection.json")["epoch"]
final_metrics = pd.read_csv(RUN / "final_metrics.csv")
final_per_class = pd.read_csv(RUN / "final_per_class.csv")
final_results = {}
for split in SPLITS:
    with np.load(RUN / f"final_{split}.npz") as stored:
        final_results[split] = {key: stored[key] for key in stored.files}
if DEVICE == "cuda":
    torch.cuda.synchronize()


## Diagnostics for the preselected report seed

These are the 09b learning curves, horizon metrics, per-class support/F1,
confusion matrices and reliability plots, using the new own-skill-conditioned
forecasts. Figures use the noninteractive Agg backend and display saved PNG bytes.
Changing `REPORT_SEED` refreshes a report; it must not become validation-based model selection.


In [ ]:
%%time
DIAGNOSTICS = RUN / "diagnostics"
DIAGNOSTICS.mkdir(exist_ok=True)

# Also select Agg here so rerunning this cell fixes an already-running kernel.
from io import BytesIO
from time import perf_counter
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
from IPython.display import Image, display

print(f"Diagnostics backend: {matplotlib.get_backend()}; output: {DIAGNOSTICS}", flush=True)

def show_figure(fig, name):
    started = perf_counter()
    def stage(message):
        print(f"[{name}, {perf_counter() - started:.2f}s] {message}", flush=True)

    try:
        stage("Laying out figure")
        # tight_layout fits labels inside the fixed canvas. A second tight-bbox
        # save would render again; inherited savefig.bbox settings are overridden.
        with matplotlib.rc_context({"savefig.bbox": None}):
            fig.tight_layout()
            stage("Rendering PNG")
            with BytesIO() as buffer:
                fig.savefig(buffer, format="png", dpi=160)
                png = buffer.getvalue()
        path = DIAGNOSTICS / f"{name}.png"
        stage("Writing PNG")
        path.write_bytes(png)
        stage("Displaying saved PNG inline")
        display(Image(data=png))  # Reuse the render; plt.show() can block on GUI backends.
        stage(f"Done: {path.name}")
    finally:
        plt.close(fig)  # Release large figures even if saving/display fails.

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for split in ["train", "test"]:
    axes[0].plot(history.epoch, history[f"{split}_nll"], label=split)
    axes[1].plot(history.epoch, history[f"{split}_macro_f1"], label=split)
for ax in axes:
    ax.axvline(best_epoch, color="black", linestyle=":", label="Selected epoch")
    ax.set_xlabel("Epoch")
    ax.legend()
axes[0].set(title="Mean horizon loss", ylabel="NLL / cross-entropy")
axes[1].set(title="Mean horizon catalogue macro-F1", ylabel="F1", ylim=(0, 1))
show_figure(fig, "learning_curves")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, measure in zip(axes, ["nll", "accuracy", "macro_f1"]):
    final_metrics.pivot(index="horizon_s", columns="split", values=measure)[list(SPLITS)].plot(
        kind="bar", ax=ax, rot=0)
    ax.set(title=f"Best checkpoint: {measure}", xlabel="Forecast horizon (s)")
    if measure != "nll": ax.set_ylim(0, 1)
show_figure(fig, "final_split_comparison")

# Plotting uses CPU arrays and does not submit CUDA work.


In [ ]:
%%time
# F1 and support for all catalogue labels, split by horizon.
fig, axes = plt.subplots(2, 3, figsize=(20, 14), sharey=True)
for h, horizon in enumerate(HORIZONS):
    subset = final_per_class.loc[final_per_class.horizon_s.eq(horizon)]
    f1 = subset.pivot(index="skill", columns="split", values="f1").reindex(
        index=data.classes, columns=SPLITS)
    counts = subset.pivot(index="skill", columns="split", values="support").reindex(
        index=data.classes, columns=SPLITS)
    image = axes[0, h].imshow(f1, vmin=0, vmax=1, aspect="auto", cmap="viridis")
    count_image = axes[1, h].imshow(np.log1p(counts), aspect="auto", cmap="magma")
    for ax in axes[:, h]:
        ax.set_xticks(range(3), SPLITS)
        ax.set_yticks(range(len(data.classes)), data.classes, fontsize=7)
    axes[0, h].set_title(f"+{horizon}s per-class F1")
    axes[1, h].set_title(f"+{horizon}s log(1 + support)")
    fig.colorbar(image, ax=axes[0, h], fraction=.04)
    fig.colorbar(count_image, ax=axes[1, h], fraction=.04)
show_figure(fig, "per_class_f1_and_support")

fig, axes = plt.subplots(3, 3, figsize=(24, 23))
for row, split in enumerate(SPLITS):
    for h, horizon in enumerate(HORIZONS):
        cm = final_results[split]["confusion"][h]
        totals = cm.sum(1, keepdims=True)
        normalized = np.divide(cm, totals, out=np.zeros_like(cm, float), where=totals > 0)
        ax = axes[row, h]
        ax.imshow(normalized, vmin=0, vmax=1, cmap="Blues")
        ax.set_xticks(range(len(data.classes)), data.classes, rotation=90, fontsize=6)
        ax.set_yticks(range(len(data.classes)), data.classes, fontsize=6)
        ax.set(title=f"{split}, +{horizon}s", xlabel="Predicted skill", ylabel="True skill")
show_figure(fig, "confusion_matrices")

# Plotting uses CPU arrays and does not submit CUDA work.


In [ ]:
%%time
# Probability calibration: each point averages one confidence bin.
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for h, horizon in enumerate(HORIZONS):
    axes[h].plot([0, 1], [0, 1], "k:", label="Perfect calibration")
    for split in SPLITS:
        result = final_results[split]
        valid = result["targets"][:, h] >= 0
        p = result["probabilities"][valid, h]
        y = result["targets"][valid, h]
        confidence, correct = p.max(-1), p.argmax(-1) == y
        bins = np.minimum((confidence * 10).astype(int), 9)
        points = [(confidence[bins == b].mean(), correct[bins == b].mean())
                  for b in range(10) if (bins == b).any()]
        axes[h].plot(*np.asarray(points).T, marker="o", label=split)
    axes[h].set(title=f"+{horizon}s reliability", xlabel="Mean confidence",
                ylabel="Observed accuracy", xlim=(0, 1), ylim=(0, 1))
    axes[h].legend()
show_figure(fig, "reliability")

with pd.option_context("display.max_rows", None):
    display(final_per_class.pivot(index=["horizon_s", "skill"], columns="split",
                                  values=["class_nll", "f1", "support"]).round(4))
print("Saved metrics, predictions, checkpoints and PNG diagnostics to:", RUN)
if MLFLOW_ENABLED:
    log_run_artifacts(RUN)  # Attach final figures and baseline tables to the same run.
if (RUN / "mlflow.json").exists():
    print("MLflow run ID:", read_json(RUN / "mlflow.json")["run_id"])

# Plotting uses CPU arrays and does not submit CUDA work.


The saved `best.pt` contains the expanded input projection, class vocabulary, feature
ordering, three horizon heads, dataset digest and model configuration. For inference,
recreate the **same causal 51-frame inputs**, including the observing aircraft's own
current skill and training-only scaling. Load with
`PairedTransformer(num_classes=len(checkpoint["classes"]), **checkpoint["model_config"])`
and `model.load_state_dict(checkpoint["model_state"])`. Never substitute the opponent's
current skill for the observer's own skill. Source code and provenance accompany every run.
